# Рейтинг курортов и его обоснование

Как из данных собирается витрина и рейтинг, из чего складывается индекс,
насколько он устойчив и почему курорт выше.
Секции: витрина resort_features → рейтинг resort_scores → компоненты,
устойчивость и объяснения.

## Витрина resort_features (Этап 4)

Собираем **одну строку на курорт** — всё, что нужно для скоринга, в одной таблице:
курортные признаки (OpenSkiMap/OSM, климат, аэропорт, координаты, ссылки Skimap/Wikidata) +
региональный спрос (поездки, рост, длительность) + цена.

Это стык трёх блоков рейтинга и основа для следующего шага (нормировка и `popularity_score`).


### 1. Из чего собирается

| Источник | Что даёт | Ключ соединения |
|---|---|---|
| `data/prepared/resorts.json` | координаты, OpenSkiMap/OSM-инфраструктура, длина/сложность трасс, климат, аэропорт, ссылки (`skimap_url`/`details_url`), качество | `resort_id` |
| `data/prepared/region_metrics.json` | поездки/ночёвки/рост/длительность (20 регионов) | `region_key` |
| `data/raw/prices_monthly.json` | цена → `adult_skipass_avg_rub` + `price_months_count` | `resort_id` |

`region_key` приводится через словарь регионов (`src/transform/regions.py`).


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root))
PROJECT_ROOT = _root

from src.transform.build_resort_features import build_resort_features  # noqa: E402

features = build_resort_features()
df = pd.DataFrame(features)
print("Строк:", len(df), "| уникальных resort_id:", df["resort_id"].nunique())
df.head()

Строк: 32 | уникальных resort_id: 32


,resort_id,name_ru,region,region_key,latitude,longitude,coordinate_quality,coordinate_status,piste_segments_6km,lift_lines_6km,...,trips_2023,trips_2024,trips_2025,nights_2023,nights_2024,nights_2025,trips_growth_2025,avg_stay_nights_2025,adult_skipass_avg_rub,price_months_count
0,R001,Роза Хутор,Краснодарский край,Краснодарский край,43.650133,40.308673,nearby_feature_proxy,updated,365,70,...,19305168.0,19163040.0,18427326.0,106687870.0,115150411.0,110670767.0,-0.038392,6.005796,6028.750000,4
1,R002,Красная Поляна,Краснодарский край,Краснодарский край,43.686110,40.260000,resort_or_complex_page,updated,121,44,...,19305168.0,19163040.0,18427326.0,106687870.0,115150411.0,110670767.0,-0.038392,6.005796,4483.333333,6
2,R003,Газпром Поляна,Краснодарский край,Краснодарский край,43.694720,40.316390,resort_or_complex_page,updated,170,54,...,19305168.0,19163040.0,18427326.0,106687870.0,115150411.0,110670767.0,-0.038392,6.005796,4100.000000,1
3,R004,Шерегеш,Кемеровская область,Кемеровская область,52.953380,87.955480,resort_or_complex_page,updated,142,23,...,1174177.0,1449604.0,1266539.0,5185409.0,4731671.0,6089657.0,-0.126286,4.808109,NaN,0
4,R005,Архыз,Карачаево-Черкесская Республика,Карачаево-Черкесская Республика,43.543330,41.181670,resort_or_complex_page,updated,42,13,...,404188.0,699802.0,1303991.0,2856842.0,2155461.0,2279411.0,0.863371,1.748027,3918.750000,4


In [ ]:
# Как агрегируется цена ски-пасса: среднее всех непустых min и max
import json as _json
from collections import defaultdict

matrix = _json.loads((PROJECT_ROOT / "data/raw/prices_monthly.json").read_text(encoding="utf-8"))
prows = [dict(zip(matrix[0], r)) for r in matrix[1:]]

vals = defaultdict(list)
for row in prows:
    rid = str(row["resort_id"])
    if row["adult_price_min_rub"] not in (None, ""):
        vals[rid].append(float(row["adult_price_min_rub"]))
    if row["adult_price_max_rub"] not in (None, ""):
        vals[rid].append(float(row["adult_price_max_rub"]))

for rid in ["R001", "R022"]:
    avg = sum(vals[rid]) / len(vals[rid])
    print(f"{rid}: {len(vals[rid])} значений min/max -> adult_skipass_avg_rub = {avg:.0f} руб")


### 2. Наши особенности — чем мы отличаемся от ТЗ/эталона

По пути мы внесли улучшения, которые отличают нашу витрину от базового ТЗ:

| Аспект | ТЗ / эталон | Мы |
|---|---|---|
| **Координаты** | один источник (Wikipedia), 26/32 неточные | 4 источника + перекрёстная проверка (Wikidata, Wikipedia, OSM-центроид, Nominatim); 30 улучшено, 2 в очереди |
| **OSM / климат / аэропорт** | посчитаны по старым координатам | пересчитаны по новым (`recompute_resort_data.py`) |
| **Сложность и типы трасс** | в ТЗ только общий счётчик трасс (`piste_segments_6km`) | добавлены из OSM: `piste:difficulty` (novice..extreme) и `piste:type` (downhill/nordic/skitour/snow_park/sled); тег сложности покрывает ~96% downhill-сегментов |
| **Модель погоды** | код запрашивает `era5_land` | исправлено на `era5` (era5_land не отдаёт `snowfall_sum` — снегопад был бы 0) |
| **region_key** | 3 алиаса | свой словарь + короткие формы республик (Кузбасс, Карачаево-Черкесия и др.) |
| **Качество** | из эталона | `completeness_pct`, `quality_flags`, `osm_overlap_warning` пересчитаны по факту |
| **Цены** | 9 курортов | оставлены 9 (вывод: нужен API агрегатора — см. `05_prices.ipynb`) |

Это позволяет показать воспроизводимость и объяснимость: каждая строка витрины
собрана из прослеживаемых источников.

### 3. Проверки

In [2]:
ids = df["resort_id"]
print("32 строки:", len(df) == 32)
print("32 уникальных resort_id:", ids.nunique() == 32)
print("нет пропусков trips_2025:", df["trips_2025"].notna().all())
print("регионов (join не размножает строки):", df["region_key"].nunique(), "(<= 20)")
print("с ценой:", df["adult_skipass_avg_rub"].notna().sum(), "| без цены:", df["adult_skipass_avg_rub"].isna().sum())

32 строки: True
32 уникальных resort_id: True
нет пропусков trips_2025: True
регионов (join не размножает строки): 20 (<= 20)
с ценой: 9 | без цены: 23


### 4. Итоговая витрина (фрагмент)

In [3]:
cols = ["resort_id", "name_ru", "region_key", "piste_segments_6km",
         "avg_snowfall_cm_per_season", "nearest_airport_distance_km",
         "trips_2025", "trips_growth_2025", "avg_stay_nights_2025",
         "adult_skipass_avg_rub", "completeness_pct"]
show = df[cols].copy()
show["trips_growth_2025"] = (show["trips_growth_2025"] * 100).round(1)
show["avg_stay_nights_2025"] = show["avg_stay_nights_2025"].round(2)
show["adult_skipass_avg_rub"] = show["adult_skipass_avg_rub"].round(0)
print(show.head(12).to_string(index=False))

resort_id        name_ru                          region_key  piste_segments_6km  avg_snowfall_cm_per_season  nearest_airport_distance_km  trips_2025  trips_growth_2025  avg_stay_nights_2025  adult_skipass_avg_rub  completeness_pct
     R001     Роза Хутор                  Краснодарский край                 365                     390.082                         36.1  18427326.0               -3.8                  6.01                 6029.0          1.000000
     R002 Красная Поляна                  Краснодарский край                 121                     390.082                         35.9  18427326.0               -3.8                  6.01                 4483.0          1.000000
     R003 Газпром Поляна                  Краснодарский край                 170                     390.082                         39.8  18427326.0               -3.8                  6.01                 4100.0          1.000000
     R004        Шерегеш                 Кемеровская область            

### Итог

- Витрина собрана и лежит в `data/mart/resort_features.csv` (скрипт
  `src/transform/build_resort_features.py`, тесты `tests/test_resort_features.py`).
- **32 строки, 32 уникальных `resort_id`**, без пропусков `trips_2025`.
- Готово к следующему шагу — нормировке 13 показателей и расчёту
  `popularity_score` (Этап 5).

## Рейтинг resort_scores (Этап 5)

Из витрины `resort_features` (32 строки) считаем составной рейтинг: нормируем
13 показателей в шкалу 0–100, применяем веса и получаем `popularity_score`,
`rank` и `confidence_score`.

### 1. Методика нормализации

| Тип | Формула | Показатели |
|---|---|---|
| Min-max | `100·(x−min)/(max−min)` | снежные/морозные дни |
| Log min-max | `100·(ln(1+x)−ln(1+min))/(ln(1+max)−ln(1+min))` | поездки, OSM, снегопад |
| Обратная | `100 − log_min_max` | аэропорт, цена |
| Рост | `clip(50 + 200·growth, 0, 100)` | рост поездок |
| Длительность | `clip((nights−1)/6·100, 0, 100)` | ночёвок на поездку |

`popularity_score = Σ (балл × вес) / 100`.

`confidence_score = 50·completeness_pct + 25·has_price + 25·has_verified_coords`
(не влияет на место, показывает качество данных).

### 2. Веса (базовая гипотеза заказчика)

| Компонент | Показатели | Вес |
|---|---|---|
| Региональный спрос | поездки 25, рост 10, длительность 5 | **40** |
| Инфраструктура | трассы 10, подъёмники 7, размещение 5, питание+достоприм. 3 | **25** |
| Зимние условия | снежные дни 8, морозные 6, снегопад 6 | **20** |
| Доступность | аэропорт 8, транспорт 2 | **10** |
| Цена | ски-пасс 5 | **5** |

Сумма = 100.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root))
PROJECT_ROOT = _root

from src.scoring.score_resorts import WEIGHTS, build_scores  # noqa: E402

rows = build_scores()
df = pd.DataFrame(rows).sort_values("rank")

print("Сумма весов:", sum(WEIGHTS.values()))
print("\nТоп-12:")
top = df.head(12)[["rank", "name_ru", "popularity_score", "confidence_score"]]
top["popularity_score"] = top["popularity_score"].round(1)
print(top.to_string(index=False))

Сумма весов: 100.0

Топ-12:
 rank        name_ru  popularity_score  confidence_score
    1 Газпром Поляна              75.5        100.000000
    2     Роза Хутор              74.5        100.000000
    3 Красная Поляна              72.8        100.000000
    4       Сорочаны              71.0         75.000000
    5          Архыз              59.5        100.000000
    6         Домбай              57.3         75.000000
    7          Игора              56.2         75.000000
    8        Шерегеш              55.6         75.000000
    9  Горный воздух              54.8         70.454545
   10   Бобровый лог              52.5         70.454545
   11         Губаха              50.5         70.454545
   12  Морозная гора              49.8         70.454545


In [ ]:
# Как нормируется показатель и складывается индекс (скелет)
import math

# 1) Log min-max на примере «трасс» (piste_segments_6km)
vals = df["piste_segments_6km"].fillna(0).astype(float).tolist()
lo, hi = math.log1p(min(vals)), math.log1p(max(vals))
rosa_val = float(df[df["resort_id"] == "R001"]["piste_segments_6km"].iloc[0])
rosa_score = 100 * (math.log1p(rosa_val) - lo) / (hi - lo)
print(f"log min-max: Роза Хутор {rosa_val:.0f} сегментов -> {rosa_score:.1f} баллов")
print(f"совпадает с готовым pistes_score: {df[df['resort_id'] == 'R001']['pistes_score'].iloc[0]:.1f}")

# 2) Взвешенная сумма 13 баллов -> индекс
r1 = df[df["resort_id"] == "R001"].iloc[0]
total = sum(r1[k + "_score"] * w for k, w in WEIGHTS.items()) / 100
print(f"Σ(балл × вес)/100 для Розы Хутор = {total:.2f} "
      f"(popularity_score в витрине: {r1['popularity_score']:.2f})")


### 3. Сверка с эталоном

In [2]:
etalon_path = (
    PROJECT_ROOT / "ТЗ" / "Travel_Designer_Resort_Ranking"
    / "data" / "processed" / "model_data.json"
)
etalon = {r["resort_id"]: r for r in json.loads(
    etalon_path.read_text(encoding="utf-8"))["model_rows"]} if etalon_path.exists() else {}

if etalon:
    cmp = []
    for _, r in df.iterrows():
        e = etalon[r["resort_id"]]
        cmp.append({
            "курорт": r["name_ru"],
            "наш": round(r["popularity_score"], 2),
            "эталон": round(e["popularity_score_default"], 2),
            "разница": round(r["popularity_score"] - e["popularity_score_default"], 2),
        })
    c = pd.DataFrame(cmp).sort_values("разница", key=abs, ascending=False)
    print("Крупнейшие расхождения (наш минус эталон):")
    print(c.head(8).to_string(index=False))
    print()
    print("Топ-12 наш   :", [r["name_ru"] for _, r in df.head(12).iterrows()])
    etalon_top = sorted(etalon.values(), key=lambda x: x["rank_default"])[:12]
    print("Топ-12 эталон:", [e["name_ru"] for e in etalon_top])
else:
    print("Эталон не найден — сверка пропущена")

Крупнейшие расхождения (наш минус эталон):
        курорт   наш  эталон  разница
       Холдоми 36.69   26.64    10.05
       Евразия 42.10   38.47     3.64
    Роза Хутор 74.50   71.57     2.93
Красная Глинка 44.87   42.06     2.82
     Завьялиха 41.04   39.00     2.04
 Горный воздух 54.83   52.79     2.04
         Чегет 49.37   47.65     1.73
Газпром Поляна 75.52   73.79     1.73

Топ-12 наш   : ['Газпром Поляна', 'Роза Хутор', 'Красная Поляна', 'Сорочаны', 'Архыз', 'Домбай', 'Игора', 'Шерегеш', 'Горный воздух', 'Бобровый лог', 'Губаха', 'Морозная гора']
Топ-12 эталон: ['Газпром Поляна', 'Красная Поляна', 'Роза Хутор', 'Сорочаны', 'Архыз', 'Домбай', 'Игора', 'Шерегеш', 'Горный воздух', 'Бобровый лог', 'Губаха', 'Эльбрус']


### 4. Наши особенности — что изменило рейтинг

Методика совпадает с эталоном (топ-3 и топ-12 почти идентичны). Расхождения
возникают там, где мы **исправили координаты** и пересчитали OpenSkiMap/OSM/климат/аэропорт:

- **Холдоми +10.05** — координата была сбита на 10 км, пересчёт по реальной точке
  поднял курорт с 32-го на 30-е место.
- **Евразия +3.64**, **Роза Хутор +2.93**, **Красная Глинка +2.82** — координаты
  уточнены к центру катания.

`confidence_score` считаем по `coordinate_validation_status` (confirmed), т.к. координаты
пересобраны по цепочке OpenSkiMap → Wikidata → Overpass (см. `02_geocoding`), а не по
старому полю `coordinate_quality`.


### 5. Обязательные проверки

In [3]:
from src.scoring.score_resorts import validate  # noqa: E402

validate(rows)  # бросит исключение при нарушении
score_cols = [k + "_score" for k in WEIGHTS]
checks = {
    "сумма весов = 100": abs(sum(WEIGHTS.values()) - 100) <= 0.001,
    "32 строки / 32 resort_id": len(df) == 32 and df["resort_id"].nunique() == 32,
    "все баллы 0..100": ((df[score_cols] >= 0) & (df[score_cols] <= 100)).all().all(),
    "popularity 0..100": df["popularity_score"].between(0, 100).all(),
    "confidence 0..100": df["confidence_score"].between(0, 100).all(),
    "ранги 1..32": sorted(df["rank"]) == list(range(1, 33)),
}
for k, v in checks.items():
    print(f"  {'OK ' if v else 'FAIL'} {k}")
assert all(checks.values())

  OK  сумма весов = 100
  OK  32 строки / 32 resort_id
  OK  все баллы 0..100
  OK  popularity 0..100
  OK  confidence 0..100
  OK  ранги 1..32


### Итог

- Рейтинг собран и лежит в `data/mart/resort_scores.csv` (скрипт
  `src/scoring/score_resorts.py`, тесты `tests/test_score_resorts.py`).
- Методика воспроизводит эталон; отличия — следствие исправленных координат.
- Готово к Этапу 6: анализ факторов, дашборд, README.

## Устойчивость и объяснение рейтинга (Этап 6)

Поверх готовой витрины `resort_scores` считаем пять укрупнённых компонентов,
их вклады в итоговый индекс, локальный рейтинг без регионального блока и
устойчивость места к весам (Монте-Карло). Плюс шаблонные объяснения «почему».

Скрипт: `src/analytics/ranking_insights.py`.
Витрины: `data/mart/resort_components.csv`, `ranking_stability.csv`,
`resort_reasons.csv`.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root))
PROJECT_ROOT = _root
from src.analytics.ranking_insights import (
    build_component_rows,
    build_reason_rows,
    build_stability_rows,
    load_scores,
)

scores = load_scores()
comp = pd.DataFrame(build_component_rows(scores))
stab = pd.DataFrame(build_stability_rows(scores))
reasons = build_reason_rows(scores)

print("компонентов (курорт × компонент):", len(comp))
print("курортов в устойчивости:", len(stab))
print("курортов в reasons:", len(reasons))


### 1. Пять компонентных баллов и вклады

Веса (сумма 100): региональный спрос 40, инфраструктура 25, зимние условия 20,
доступность 10, цена 5. Компонентный балл — взвешенная средняя входящих баллов;
вклад = балл × вес / 100. Сумма пяти вкладов равна индексу потенциала.


In [ ]:
# Проверка: сумма вкладов == индекс потенциала
chk = comp.groupby("resort_id")["contribution"].sum().reset_index()
m = stab[["resort_id", "popularity_score"]].merge(chk, on="resort_id")
m["delta"] = (m["contribution"] - m["popularity_score"]).abs()
print("Макс. расхождение суммы вкладов с индексом:", m["delta"].max())

# Пример: компоненты одного курорта
print()
print(comp[comp["resort_id"] == "R001"][["component_name_ru", "component_score", "contribution"]].to_string(index=False))


### 2. Локальный рейтинг без регионального блока

`local_score = (popularity_score − региональный_вклад) / 0.60`.
Показывает, курорт силён сам по себе или в основном за счёт масштаба региона.


In [ ]:
cols = ["rank", "name_ru", "popularity_score", "regional_contribution",
        "local_rank", "median_rank", "rank_p5", "rank_p95", "top10_frequency"]
show = stab[cols].head(12).copy()
show["popularity_score"] = show["popularity_score"].round(1)
show["top10_frequency"] = show["top10_frequency"].round(2)
print(show.to_string(index=False))


In [ ]:
# Эффект регионального блока: Сорочаны (#4) и Горный воздух (#9)
for rid in ["R030", "R016"]:
    r = stab[stab["resort_id"] == rid].iloc[0]
    print(f"{r['name_ru']}: место {r['rank']} -> локальное {r['local_rank']} "
          f"(региональный вклад {r['regional_contribution']:.1f})")


### 3. Устойчивость к весам (Монте-Карло)

5000 сценариев весов (мультипликативный шум ×U(0.6, 1.4) вокруг базы, затем
перенормировка к 100), фиксированный seed. Медианное место, диапазон мест
(5–95-й процентили), частота топ-10. Это «диапазон мест в сценариях весов»,
а не доверительный интервал качества курорта.


In [ ]:
# Скелет Монте-Карло: как варьируются веса и пересчитывается индекс
import random

BASE = [40.0, 25.0, 20.0, 10.0, 5.0]  # веса 5 компонентов
ORDER = ["regional", "infrastructure", "winter", "accessibility", "price"]

# компонентные баллы Розы Хутор (R001) из уже загруженного comp
scores = comp[comp["resort_id"] == "R001"].set_index("component")["component_score"].to_dict()

rng = random.Random(42)  # фиксированный seed -> воспроизводимость
print("5 примеров сценариев весов для Розы Хутор (всего их 5000):")
for _ in range(5):
    noisy = [w * rng.uniform(0.6, 1.4) for w in BASE]   # мультипликативный шум
    total = sum(noisy)
    weights = [w / total * 100 for w in noisy]          # нормировка к сумме 100
    index = sum(scores[c] * w for c, w in zip(ORDER, weights)) / 100
    print(f"  веса {[round(w) for w in weights]} -> индекс {index:.1f}")
print("Дальше так же 5000 раз; для каждого курорта фиксируется место, "
      "а медиана и диапазон уже лежат в ranking_stability.")


In [ ]:
# Устойчивые (топ-3) против чувствительных (широкий диапазон мест)
top3 = stab.head(3)["name_ru"].tolist()
sens = stab.sort_values("rank_p95", ascending=False).head(3)["name_ru"].tolist()
print("Устойчивые (топ-3):", top3)
print("Наиболее чувствительные (широкий диапазон мест):", sens)


### 4. Сценарии «климат важнее» и «доступность важнее»

Явные сценарии: зимние условия 20 → 30 (за счёт региона) и доступность 10 → 15.
Место курорта в этих сценариях хранится в `ranking_stability`.


In [ ]:
cols = ["rank", "name_ru", "rank_climate_more", "rank_accessibility_more"]
show = stab[cols].copy()
show["d_climate"] = show["rank"] - show["rank_climate_more"]
show["d_access"] = show["rank"] - show["rank_accessibility_more"]
top = show.sort_values("d_climate", ascending=False).head(5)
print("Кто поднимается при «климат важнее» (положительное d_climate):")
print(top[["name_ru", "rank", "rank_climate_more", "d_climate"]].to_string(index=False))


### 5. Объяснения (reason_codes)

Шаблонные объяснения трёх типов: `strength` (сильные стороны), `caveat`
(что проверить), `tradeoff` (компромисс). Генерируются правилами, без ИИ.


In [ ]:
for rid in ["R003", "R030", "R016", "R014"]:
    row = next(r for r in reasons if r["resort_id"] == rid)
    print(f"{row['name_ru']}:")
    for x in row["reasons"]:
        print(f"  [{x['type']}] {x['text']}")
    print()


### Итог

- Топ-3 устойчивы (диапазон мест 1–3, топ-10 в 100% сценариев).
- Сорочаны (#4) держатся регионом (Московская область) и без него падают.
- Горный воздух (#9) силён сам по себе (доступность).
- Компоненты и вклады дают объяснимую структуру индекса для дашборда.
